# Analyse linguistique des 4 cas (P1 Finetuné vs KATE)

On analyse les **4 sous-ensembles** :
- **Les 2 se trompent** (00)
- **Les 2 ont raison** (11)
- **P1✓ KATE✗** (10) — complémentarité : finetuning seul correct
- **P1✗ KATE✓** (01) — complémentarité : few-shot seul correct

**Features calculées** :
1. **Chevauchement lexical** : % de mots en commun (prémisse ↔ hypothèse) — piège classique NLI (fort overlap → prédire Entailment à tort).
2. **Densité numérique** : chiffres, %, unités (mg, ans, etc.) — les essais cliniques ont beaucoup de seuils ; le few-shot peut être moins bon sur le raisonnement numérique.
3. **Négations** : "non", "aucun", "sans", "ne pas" — contradiction par négation simple vs antonymie complexe.
4. **Autres** : longueur (prémisse/hypothèse), type Single vs Comparison, vocabulaire "primary/secondary trial".

Objectif : voir si on peut **distinguer** ces 4 cas et en tirer des **conclusions** (où chaque approche est forte ou faible).

## 1. Configuration et chargement des données

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

NLI4CT_ROOT = Path(".").resolve()
if not (NLI4CT_ROOT / "results").exists():
    NLI4CT_ROOT = NLI4CT_ROOT / "NLI4CT"
RESULTS_DIR_P1 = NLI4CT_ROOT / "results" / "Prompt 1"
RESULTS_DIR_KATE = NLI4CT_ROOT / "results" / "Fewshot_Kate"
FIGURES_DIR = NLI4CT_ROOT / "results" / "compare_figures_ft_vs_kate"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
GOLD_TEST_JSON = NLI4CT_ROOT / "Gold_test.json"
GOLD_JSONL_P1 = RESULTS_DIR_P1 / "Gold_test_formatted_old_prompt.jsonl" if (RESULTS_DIR_P1 / "Gold_test_formatted_old_prompt.jsonl").exists() else RESULTS_DIR_P1 / "Gold_test_formatted.jsonl"
CSV_FT_1 = next(RESULTS_DIR_P1.glob("pred_ft_*.csv"), None)
CSV_KATE = next(RESULTS_DIR_KATE.glob("pred_fewshot_KATE*.csv"), None)
assert CSV_FT_1 and CSV_KATE, "CSV P1 ou KATE manquant."

In [ ]:
def extract_premise_hypothesis(text):
    if "HYPOTHESIS:" not in text:
        return text[:800] if len(text) > 800 else text, text[:500] if len(text) > 500 else text
    before, after = text.split("HYPOTHESIS:", 1)
    premise = before.replace("PREMISE:", "").strip()
    for sep in (".\n\nAnswer", "?\n\nAnswer", "? Answer", "\n\nAnswer"):
        if sep in after: after = after.split(sep)[0]
    return premise, after.strip()

df_ft_1 = pd.read_csv(CSV_FT_1)
df_kate = pd.read_csv(CSV_KATE)
for df in (df_ft_1, df_kate):
    if df['is_correct'].dtype == object:
        df['is_correct'] = df['is_correct'].apply(lambda x: str(x).strip().lower() == 'true')

meta_list = []
with open(GOLD_JSONL_P1, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if not line.strip(): continue
        obj = json.loads(line)
        msgs = obj.get("messages", [])
        if len(msgs) < 2: continue
        user_content = ""
        for m in msgs:
            if m.get("role") == "user": user_content = m.get("content", "")
        prem, stmt = extract_premise_hypothesis(user_content)
        meta_list.append({'index': idx, 'premise': prem, 'hypothesis': stmt})
df_meta = pd.DataFrame(meta_list)

common_idx = list(set(df_ft_1['index']) & set(df_kate['index']))
ok_p1 = df_ft_1.set_index('index').loc[common_idx, 'is_correct'].reindex(common_idx).fillna(False).values
ok_kate = df_kate.set_index('index').loc[common_idx, 'is_correct'].reindex(common_idx).fillna(False).values
acc = pd.DataFrame({'index': common_idx, 'P1_ft': ok_p1, 'KATE': ok_kate})
acc['pattern'] = acc['P1_ft'].astype(int).astype(str) + acc['KATE'].astype(int).astype(str)

df_base = df_meta[df_meta['index'].isin(common_idx)].copy()
df_base = df_base.merge(acc[['index','pattern']], on='index', how='inner')
if GOLD_TEST_JSON.exists():
    with open(GOLD_TEST_JSON, 'r', encoding='utf-8') as f: gold = json.load(f)
    keys = list(gold.keys())
    type_sec = pd.DataFrame([{'index': i, 'type': gold[keys[i]].get('Type','N/A'), 'section_id': gold[keys[i]].get('Section_id','N/A')} for i in range(len(keys))])
    df_base = df_base.merge(type_sec, on='index', how='left')

pattern_labels = {'11': 'Les 2 corrects', '00': 'Les 2 en erreur', '10': 'P1✓ KATE✗', '01': 'P1✗ KATE✓'}
df_base['pattern_label'] = df_base['pattern'].map(pattern_labels)
print('Effectifs par cas:')
print(df_base['pattern_label'].value_counts().sort_index())

## 2. Fonctions d’analyse linguistique

In [ ]:
def tokenize(text):
    """Mots en minuscules, alphanumeriques."""
    if pd.isna(text) or not isinstance(text, str): return set()
    return set(re.findall(r"[a-zA-Z0-9À-ÿ]+", text.lower()))

def lexical_overlap(premise, hypothesis):
    """Jaccard + % mots de l'hypothèse présents dans la prémisse (coverage)."""
    p, h = tokenize(premise), tokenize(hypothesis)
    if not p and not h: return 0.0, 0.0
    inter = len(p & h)
    union = len(p | h)
    jaccard = inter / union if union else 0.0
    coverage = inter / len(h) if h else 0.0  # % des mots de l'hypothèse dans la prémisse
    return jaccard, coverage

def numeric_density(text):
    """Nombre de chiffres, de pourcentages (\d+%), et d'unités (mg, g, ml, ans, years, etc.)."""
    if pd.isna(text) or not isinstance(text, str): return 0, 0, 0
    digits = len(re.findall(r"\d", text))
    pct = len(re.findall(r"\d+%", text))
    units = len(re.findall(r"\d+\s*(?:mg|g|ml|mL|kg|years?|yrs?|months?|days?|weeks?|cycles?|doses?|mg/m²|m²)", text, re.I))
    return digits, pct, units

def negation_presence(text):
    """Compte d'occurrences de négations (non, aucun, sans, ne pas, ni, jamais, etc.)."""
    if pd.isna(text) or not isinstance(text, str): return 0
    t = text.lower()
    neg = re.findall(r"\b(?:non|aucun|aucune|sans|ni\b|jamais|pas\b|n'est|n'ont|ne pas|ne pas|not\b|no\b|none|never|cannot|can't)", t)
    return len(neg)

def trial_keywords(text):
    """Présence de mots clés 'primary' / 'secondary' (trial)."""
    if pd.isna(text) or not isinstance(text, str): return 0, 0
    t = text.lower()
    primary = len(re.findall(r"primary\s+(?:trial)?", t))
    secondary = len(re.findall(r"secondary\s+(?:trial)?", t))
    return primary, secondary

def word_count(text):
    if pd.isna(text) or not isinstance(text, str): return 0
    return len(re.findall(r"[a-zA-Z0-9À-ÿ]+", text))

In [ ]:
# Appliquer les features à chaque ligne
def compute_features(row):
    prem, hyp = row['premise'], row['hypothesis']
    jaccard, coverage = lexical_overlap(prem, hyp)
    digits_p, pct_p, units_p = numeric_density(prem)
    digits_h, pct_h, units_h = numeric_density(hyp)
    neg_p = negation_presence(prem)
    neg_h = negation_presence(hyp)
    prim_p, sec_p = trial_keywords(prem)
    prim_h, sec_h = trial_keywords(hyp)
    return pd.Series({
        'lexical_jaccard': jaccard,
        'lexical_coverage': coverage,
        'digits_premise': digits_p,
        'digits_hypothesis': digits_h,
        'pct_premise': pct_p,
        'pct_hypothesis': pct_h,
        'units_premise': units_p,
        'units_hypothesis': units_h,
        'numeric_total': digits_p + digits_h + units_p + units_h + pct_p + pct_h,
        'neg_premise': neg_p,
        'neg_hypothesis': neg_h,
        'neg_total': neg_p + neg_h,
        'words_premise': word_count(prem),
        'words_hypothesis': word_count(hyp),
        'trial_primary': prim_p + prim_h,
        'trial_secondary': sec_p + sec_h,
    })

df_feat = df_base.join(df_base.apply(compute_features, axis=1))
df_feat.head(3)

## 3. Statistiques descriptives par cas (4 groupes)

In [ ]:
feat_cols = ['lexical_jaccard', 'lexical_coverage', 'numeric_total', 'neg_total', 'words_premise', 'words_hypothesis']
summary = df_feat.groupby('pattern_label')[feat_cols].agg(['mean', 'median', 'std', 'count']).round(4)
display(summary)
print('\n--- Moyennes par cas (pour comparaison rapide) ---')
means = df_feat.groupby('pattern_label')[feat_cols].mean().round(4)
display(means)

## 4. Visualisations par feature (boxplots / barres)

In [ ]:
order = ['Les 2 corrects', 'Les 2 en erreur', 'P1✓ KATE✗', 'P1✗ KATE✓']
df_plot = df_feat[df_feat['pattern_label'].isin(order)].copy()
df_plot['pattern_label'] = pd.Categorical(df_plot['pattern_label'], categories=order, ordered=True)
df_plot = df_plot.sort_values('pattern_label')

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()
for i, col in enumerate(feat_cols):
    ax = axes[i]
    sns.boxplot(data=df_plot, x='pattern_label', y=col, ax=ax, order=order)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.set_title(col)
axes[-1].axis('off')
plt.suptitle('Distribution des features linguistiques par cas (P1 FT vs KATE)', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'linguistic_boxplots_4_cas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Moyennes en barres pour lecture plus directe
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(feat_cols):
    ax = axes[i]
    m = df_feat.groupby('pattern_label')[col].mean().reindex(order).dropna()
    m.plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c','#3498db','#9b59b6'], edgecolor='black')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
    ax.set_title(col)
    ax.set_ylabel('Moyenne')
axes[-1].axis('off')
plt.suptitle('Moyennes des features par cas', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'linguistic_means_4_cas.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Analyse ciblée : Chevauchement lexical

In [ ]:
print('Hypothèse : fort chevauchement lexical → les modèles prédiraient Entailment à tort (contradiction subtile).')
print('On compare Jaccard et coverage entre "Les 2 en erreur" vs "Les 2 corrects".\n')
for label in ['Les 2 en erreur', 'Les 2 corrects']:
    sub = df_feat[df_feat['pattern_label'] == label]
    print(f"{label}: Jaccard moyen = {sub['lexical_jaccard'].mean():.3f}, coverage moyen = {sub['lexical_coverage'].mean():.3f} (n={len(sub)})")
print()
print('Comparaison P1✓ KATE✗ vs P1✗ KATE✓ (complémentarité) :')
for label in ['P1✓ KATE✗', 'P1✗ KATE✓']:
    sub = df_feat[df_feat['pattern_label'] == label]
    print(f"{label}: Jaccard = {sub['lexical_jaccard'].mean():.3f}, coverage = {sub['lexical_coverage'].mean():.3f}")

## 6. Analyse ciblée : Densité numérique

In [ ]:
print('Hypothèse : cas avec plus de chiffres/unités → KATE (few-shot) pourrait plus souvent se tromper (raisonnement numérique).')
print('On compare numeric_total (chiffres + % + unités) entre les 4 cas.\n')
for label in order:
    sub = df_feat[df_feat['pattern_label'] == label]
    print(f"{label}: numeric_total moyen = {sub['numeric_total'].mean():.1f}, units_premise = {sub['units_premise'].mean():.1f}, units_hypothesis = {sub['units_hypothesis'].mean():.1f}")
print()
print('Si P1✗ KATE✓ a une densité numérique plus élevée → KATE gère mieux le numérique dans ces exemples.')

## 7. Analyse ciblée : Négations

In [ ]:
print('Hypothèse : plus de négations → contradiction par négation explicite vs antonymie/complexe.')
print('Comparaison neg_total (prémisse + hypothèse) entre les 4 cas.\n')
for label in order:
    sub = df_feat[df_feat['pattern_label'] == label]
    print(f"{label}: neg_total moyen = {sub['neg_total'].mean():.2f}, neg_hypothesis = {sub['neg_hypothesis'].mean():.2f}")
print()
print('Vérification par type (Single vs Comparison) :')
if 'type' in df_feat.columns:
    display(df_feat.groupby(['pattern_label','type']).size().unstack(fill_value=0))

## 9. Synthèse et conclusions

In [ ]:
conclusions = []
for label in order:
    sub = df_feat[df_feat['pattern_label'] == label]
    conclusions.append({
        'Cas': label,
        'n': len(sub),
        'Jaccard_moy': sub['lexical_jaccard'].mean(),
        'numeric_moy': sub['numeric_total'].mean(),
        'neg_moy': sub['neg_total'].mean(),
        'words_prem_moy': sub['words_premise'].mean(),
    })
df_concl = pd.DataFrame(conclusions)
display(df_concl.round(4))

print('\n--- Interprétation possible ---')
print('• Les 2 en erreur : si Jaccard élevé → piège lexical (overlap). Si numeric élevé → difficulté numérique partagée.')
print('• Les 2 corrects : baseline "facile" ; à comparer aux autres pour voir quelles features distinguent le facile du difficile.')
print('• P1✓ KATE✗ : cas où le finetuning réussit et KATE échoue ; si numeric élevé → finetuning plus robuste au numérique.')
print('• P1✗ KATE✓ : cas où KATE réussit et le finetuning échoue ; si neg ou lexical particulier → KATE meilleur sur ces profils.')

## 8b. Test de significativité (Kruskal-Wallis)

Vérifier si les distributions des features diffèrent significativement entre les 4 groupes.

In [ ]:
from scipy import stats
groups = [df_feat.loc[df_feat['pattern_label']==l, col].values for l in order for col in feat_cols]
results = []
for col in feat_cols:
    g = [df_feat.loc[df_feat['pattern_label']==l, col].dropna().values for l in order]
    g = [x for x in g if len(x) > 0]
    if len(g) >= 2:
        h, p = stats.kruskal(*g)
        results.append({'feature': col, 'H': h, 'p_value': p})
df_kw = pd.DataFrame(results)
df_kw['significatif_5%'] = df_kw['p_value'] < 0.05
display(df_kw.round(4))
print('Si p_value < 0.05 : les 4 groupes diffèrent significativement sur cette feature.')

## 10. Export du tableau enrichi

In [ ]:
out_path = FIGURES_DIR / 'analyse_linguistique_4_cas.csv'
df_feat.to_csv(out_path, index=False)
print('Exporté:', out_path)